# Phase 1: Vehicle Detection and Tracking (Smart Traffic Violation Detection System)

Welcome to Phase 1 of the **Smart Traffic Violation Detection System**! This notebook is designed to run in Google Colab.
In this notebook, we will install YOLOv8 and ByteTrack (via `ultralytics`), upload a traffic video, detect and track vehicles, and save the result as an H.264-encoded web-compatible MP4.

### Step 1: Install Required Libraries
First, we need to install the `ultralytics` package, which houses both **YOLOv8** and the built-in, optimized **ByteTrack** engine. We will also verify if a GPU is allocated to speed up processing.

In [ ]:
# Install the Ultralytics YOLOv8 library
!pip install -q ultralytics

# Verify GPU support and imports
import torch
import ultralytics
from ultralytics import YOLO
import cv2
import os

print(f"CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected. Running on CPU. (To enable GPU, go to Edit -> Notebook Settings -> Hardware Accelerator -> select GPU T4)")

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

### Step 2: Upload CCTV Traffic Video
Now, upload your traffic CCTV video (`.mp4` format). Colab will prompt you to select a file from your computer. We will automatically rename it to `input_video.mp4` for consistent processing.

In [ ]:
from google.colab import files

print("Please select your traffic video file (.mp4) to upload:")
uploaded = files.upload()

# Get the name of the uploaded file
uploaded_filename = list(uploaded.keys())[0]

# Ensure it's an MP4 file
if not uploaded_filename.lower().endswith('.mp4'):
    raise ValueError(f"Uploaded file '{uploaded_filename}' is not a .mp4 video. Please run this cell again and select an MP4 video.")

# Rename the file to a standard input name
input_video_path = 'input_video.mp4'
if os.path.exists(input_video_path):
    os.remove(input_video_path)

os.rename(uploaded_filename, input_video_path)
print(f"Successfully uploaded and saved video as '{input_video_path}'")

### Step 3: Detection, Tracking, and HUD Overlay Pipeline
In this step, we run the tracking pipeline.
1. **Model Loading:** We load the YOLOv8-Nano (`yolov8n.pt`) model. It's lightweight and super fast, perfect for real-time processing in Colab.
2. **COCO Filters:** We restrict our detections to Cars, Motorcycles, Buses, and Trucks.
3. **ByteTrack:** We use the built-in tracking module which keeps track of the vehicle ID across frames.
4. **Custom HUD:** We draw a semi-transparent black overlay in the top-left corner to show active vehicles, total unique vehicles logged, and counts by category.
5. **Video Compilation:** We save the video, then run `ffmpeg` to re-encode it with the browser-compatible H.264 codec.

In [ ]:
# 1. Initialize YOLOv8 Model
# yolov8n.pt is pre-trained on COCO dataset which includes cars, buses, trucks, and motorcycles
model = YOLO('yolov8n.pt')

input_video_path = 'input_video.mp4'
temp_output_path = 'output_tracked_raw.mp4'
final_output_path = 'output_tracked.mp4'

if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"'{input_video_path}' not found. Please upload a video in Step 2 first.")

# 2. Open Video Stream
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    raise ValueError("Error: Could not open the video file.")

# Extract video dimensions, frame rate, and frame count
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Set up raw VideoWriter (mp4v codec)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_output_path, fourcc, fps, (width, height))

# 3. Class Definitions & Style Setup
CLASS_NAMES = {2: 'Car', 3: 'Motorcycle', 5: 'Bus', 7: 'Truck'}
TARGET_CLASSES = list(CLASS_NAMES.keys())

# Harmonious HSL-like colors for drawing (B, G, R format for OpenCV)
CLASS_COLORS = {
    2: (255, 102, 102),   # Coral Red for Cars
    3: (102, 255, 102),   # Mint Green for Motorcycles
    5: (102, 178, 255),   # Sky Blue for Buses
    7: (255, 178, 102)    # Warm Orange for Trucks
}

# Cumulative Stats Counters
total_tracked_vehicles = set()
vehicle_type_stats = {name: set() for name in CLASS_NAMES.values()}

print("Pipeline started processing. This may take a few minutes depending on video length...")

# 4. Stream-based Tracking Generator Loop
results = model.track(
    source=input_video_path,
    tracker="bytetrack.yaml",
    persist=True,
    classes=TARGET_CLASSES,
    stream=True
)

frame_count = 0

for result in results:
    frame = result.orig_img.copy()
    boxes = result.boxes
    active_vehicle_count = 0
    
    # Process detections if active tracks exist
    if boxes is not None and boxes.id is not None:
        track_ids = boxes.id.int().cpu().tolist()
        class_ids = boxes.cls.int().cpu().tolist()
        confidences = boxes.conf.cpu().tolist()
        xyxys = boxes.xyxy.int().cpu().tolist()
        
        active_vehicle_count = len(track_ids)
        
        # Draw trackers & update metrics
        for track_id, class_id, conf, xyxy in zip(track_ids, class_ids, confidences, xyxys):
            class_name = CLASS_NAMES.get(class_id, 'Vehicle')
            
            # Register unique track IDs
            total_tracked_vehicles.add(track_id)
            vehicle_type_stats[class_name].add(track_id)
            
            # Look up color
            color = CLASS_COLORS.get(class_id, (0, 255, 255))
            
            # Draw bounding box
            x1, y1, x2, y2 = xyxy
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            
            # Text Tag
            label = f"{class_name} ID:{track_id} {conf:.2f}"
            
            # Draw text backplate
            font_face = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            thickness = 2
            text_size = cv2.getTextSize(label, font_face, font_scale, thickness)[0]
            
            label_y = max(y1, text_size[1] + 10)
            cv2.rectangle(frame, (x1, label_y - text_size[1] - 8), (x1 + text_size[0] + 6, label_y), color, -1)
            cv2.putText(frame, label, (x1 + 3, label_y - 4), font_face, font_scale, (0, 0, 0), thickness)
            
    # 5. Overlay Semi-Transparent HUD
    hud_overlay = frame.copy()
    cv2.rectangle(hud_overlay, (15, 15), (320, 220), (0, 0, 0), -1)
    cv2.addWeighted(hud_overlay, 0.65, frame, 0.35, 0, frame)
    
    # Render HUD text
    cv2.putText(frame, "TRAFFIC MONITOR HUD", (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.line(frame, (25, 48), (310, 48), (180, 180, 180), 1)
    
    cv2.putText(frame, f"Active Vehicles: {active_vehicle_count}", (25, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    cv2.putText(frame, f"Total Logged: {len(total_tracked_vehicles)}", (25, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)
    
    # Write stats breakdown
    stats_y = 130
    for name, unique_ids in vehicle_type_stats.items():
        count = len(unique_ids)
        cv2.putText(frame, f"- {name}s: {count}", (35, stats_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)
        stats_y += 20

    # Write output frame
    out.write(frame)
    
    # Periodic progress prints
    frame_count += 1
    if frame_count % 30 == 0 or frame_count == total_frames:
        print(f"Processed Frame {frame_count}/{total_frames} ({(frame_count/total_frames)*100:.1f}%)")

# Clean up resources
cap.release()
out.release()

# 5. Define final statistics variables for downstream dashboard integration
total_vehicle_count = len(total_tracked_vehicles)
vehicle_type_count = {name: len(unique_ids) for name, unique_ids in vehicle_type_stats.items()}

print("Tracking completed! Re-encoding raw video to browser-native H.264 codec...")

# 6. Re-encode video file using pre-installed ffmpeg
if os.path.exists(temp_output_path):
    if os.path.exists(final_output_path):
        os.remove(final_output_path)
    
    # Re-encode to H.264 and delete raw output
    os.system(f"ffmpeg -y -i {temp_output_path} -vcodec libx264 {final_output_path} -loglevel warning")
    os.remove(temp_output_path)
    print(f"H.264 Conversion successful! Output video saved to: {final_output_path}")
else:
    print("Error: Tracking failed. The raw video could not be compiled.")

### Step 4: Display Results & Statistics
Now that the video is processed, we can review the final traffic logging statistics and play the tracked video directly inside Colab!

> [!NOTE]
> **Large Video Note:** If your video file is very large (over 20-30MB), encoding and playing it inside the browser cell might take a moment. If the browser stalls, you can also download the `output_tracked.mp4` file directly using the File Explorer pane on the left side of Google Colab.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# 1. Print beautifully formatted Summary Statistics
print("=" * 40)
print("       FINAL TRAFFIC ANALYSIS REPORT   ")
print("=" * 40)
print(f"Total Unique Vehicles Logged: {total_vehicle_count}")
print("-" * 40)
for name, count in vehicle_type_count.items():
    print(f"   - {name}s Detected: {count}")
print("=" * 40)

# 2. Embed the Video directly in Google Colab
if os.path.exists('output_tracked.mp4'):
    # Load and encode video
    mp4 = open('output_tracked.mp4', 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    
    # Restrict output size for a clean look
    html_code = f"""
    <div style="text-align: center; margin-top: 10px;">
        <video width="854" height="480" controls style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <source src="{{data_url}}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
    </div>
    """
    # Note: Using python string formatting we escape double braces to keep f-string placeholder safe
    html_code = html_code.replace('{{data_url}}', data_url)
    display(HTML(html_code))
else:
    print("Error: output_tracked.mp4 was not found. Please ensure Step 3 ran without errors.")